# CSE 3104 — Final Project: Fast Food & Income Pipeline

**Team:** Andrew, Imaan, Juliet  
**Date:** April 2026  

---

This notebook builds the master ZIP-level table combining Census income, population, and restaurant location data for all five chains.

**Steps:**
1. Imports & folder setup
2. Load ACS Census income data (B19013)
3. Impute missing income values
4. Load ACS population data (DP05)
5. Load all restaurant location files and count per ZIP
6. Build the master table
7. Calculate store density per capita
8. Export final output
9. Quick sanity checks

## Step 1 — Imports

In [ ]:
# libraries we need
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# this notebook sits in New Results/ so source_data and outputs are right next to it
BASE_DIR = os.getcwd()

SOURCE_DIR = os.path.join(BASE_DIR, 'source_data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

def source(filename):
    return os.path.join(SOURCE_DIR, filename)

def output(filename):
    return os.path.join(OUTPUT_DIR, filename)

print('Source data folder:', SOURCE_DIR)
print('Outputs folder:    ', OUTPUT_DIR)
print('Source files:', os.listdir(SOURCE_DIR))

## Step 2 — Census Income Data (ACS B19013)

Load ACS 5-Year median household income by ZIP code. Missing values are imputed using the state average.

In [ ]:
# load the census income file, skip the extra header row Census adds
census = pd.read_csv(source('ACSDT5Y2024.B19013-Data.csv'), skiprows=[1])

census['zip_code'] = census['NAME'].str.extract(r'(\d{5})') # pull just the 5 digit zip
census['median_income'] = pd.to_numeric(census['B19013_001E'], errors='coerce') # make sure income is a number
census = census[['zip_code', 'median_income']].copy() # only keep what we need

print(f'Census rows loaded: {len(census):,}')
print(f'Missing income values: {census["median_income"].isna().sum():,}')
census.head(3)

## Step 3 — Impute Missing Income

In [ ]:
# some zips are missing income data, fill them in using the average of nearby zips in the same state
census['state_prefix'] = census['zip_code'].str[:2] # first 2 digits = state

missing_before = census['median_income'].isna().sum()

census['median_income'] = census.groupby('state_prefix')['median_income'].transform(
    lambda x: x.fillna(x.mean())
)

missing_after = census['median_income'].isna().sum()
print(f'Imputed {missing_before - missing_after} values ({missing_before} -> {missing_after} remaining missing)')

census = census.drop(columns='state_prefix') # dont need this anymore
census.head(3)

## Step 4 — Population Data (ACS DP05)

Load ZIP-level total population. Used in Step 7 to normalize store counts into stores per 10,000 residents.

In [ ]:
# load ZIP-level population from the DP05 file, same skip pattern as the income file
demographics_raw = pd.read_csv(source('ACSDP5Y2024.DP05-Data.csv'), skiprows=[1], low_memory=False)

demographics = demographics_raw[['NAME', 'DP05_0001E']].copy()
demographics['zip_code'] = demographics['NAME'].str.extract(r'(\d{5})') # pull the zip
demographics['total_population'] = pd.to_numeric(demographics['DP05_0001E'], errors='coerce') # make sure its a number
demographics = demographics[['zip_code', 'total_population']].dropna(subset=['zip_code'])

print(f'Population rows loaded: {len(demographics):,}')
demographics.head(3)

## Step 5 — Restaurant Location Files

Load all five restaurant location CSVs and count locations per ZIP code.

- McDonald's (Andrew)
- Chipotle (Imaan + Juliet)
- Panera Bread (Juliet)
- Five Guys (Juliet)
- Panda Express (Juliet)

In [ ]:
# reusable function to load any restaurant file and count locations per zip
def count_per_zip(filepath, zip_col, count_col_name):
    df = pd.read_csv(filepath)
    df['zip_code'] = df[zip_col].astype(str).str.zfill(5).str[:5] # standardize to 5 digit zip
    counts = (
        df.groupby('zip_code')
        .size()
        .reset_index(name=count_col_name)
    )
    print(f'{count_col_name}: {len(df):,} locations across {len(counts):,} unique ZIPs')
    return counts


mcdonalds_counts = count_per_zip(source('mcdonalds_locations.csv'),     'zipcode', 'mcdonalds_count')
chipotle_counts  = count_per_zip(source('chipotle_locations.csv'),      'zipcode', 'chipotle_count')
panera_counts    = count_per_zip(source('panera_locations.csv'),        'zipcode', 'panera_count')
five_guys_counts = count_per_zip(source('five_guys_locations.csv'),     'zipcode', 'five_guys_count')
panda_counts     = count_per_zip(source('panda_express_locations.csv'), 'zipcode', 'panda_express_count')

## Step 6 — Build the Master Table

Start with all Census ZIP codes as the base, then left-join each chain's counts. ZIP codes with no location for a given chain get 0.

In [ ]:
# start with census as the base so we keep all ~33k zip codes as rows
master = census.copy()
master = master.merge(demographics, on='zip_code', how='left') # add population

# add each restaurant's count, zips with no location for that chain get 0
for counts_df in [mcdonalds_counts, chipotle_counts, panera_counts, five_guys_counts, panda_counts]:
    master = master.merge(counts_df, on='zip_code', how='left')

count_cols = ['mcdonalds_count', 'chipotle_count', 'panera_count', 'five_guys_count', 'panda_express_count']
master[count_cols] = master[count_cols].fillna(0).astype(int)

print(f'Master table shape: {master.shape}')
master.head(5)

In [ ]:
# how many ZIP codes have at least one location for each chain
print('ZIP codes with at least one location per chain:')
for col in count_cols:
    covered = (master[col] > 0).sum()
    total = len(master)
    print(f'  {col:<25} {covered:>6,}  ({covered/total*100:.1f}% of all ZIPs)')

## Step 7 — Store Density Per Capita

Normalize store counts to stores per 10,000 residents so ZIP codes of different sizes are comparable.

In [ ]:
# normalize store counts by population so we can compare zips fairly
density_cols = []

valid_pop = master['total_population'] > 0 # skip zips with zero or missing population
for col in count_cols:
    density_col = col.replace('_count', '_density')
    master[density_col] = np.where(
        valid_pop,
        (master[col] / master['total_population']) * 10000, # stores per 10k residents
        np.nan
    )
    density_cols.append(density_col)

print('Density columns added (stores per 10,000 residents):')
print(master[density_cols].describe().round(4))

## Step 8 — Export Final Output

In [ ]:
# save the master table to outputs/
master.to_csv(output('master_pipeline_output.csv'), index=False)
print(f'Saved  ({len(master):,} rows, {len(master.columns)} columns)')
print(f'Columns: {master.columns.tolist()}')

## Step 9 — Quick Sanity Checks

In [ ]:
# quick checks to make sure the pipeline output looks right
print(f'Total ZIP codes: {len(master):,}')

print('Missing values per column:')
print(master.isnull().sum())

print('Store count totals:')
for col in count_cols:
    print(f'  {col}: {master[col].sum():,}')

print(f'Income range: ${master["median_income"].min():,.0f} - ${master["median_income"].max():,.0f}  '
      f'(mean: ${master["median_income"].mean():,.0f})')

# sample zips where at least 3 chains are present
master[
    (master['mcdonalds_count'] > 0) &
    (master['chipotle_count'] > 0) &
    (master['panera_count'] > 0)
][['zip_code', 'median_income', 'total_population'] + count_cols].head(5)